# 미로 생성 AI 알고리즘

재귀적 백트래킹(Recursive Backtracking)으로 미로를 생성하고, BFS로 최단 경로를 찾습니다.

**실행 순서**: 위에서 아래로 셀을 차례대로 실행하세요 (Shift + Enter)

## 셀 1 · 한글 폰트 설치 (한 번만 실행)

실행 후 **런타임 → 세션 다시 시작**을 한 번 해주면 한글이 깨지지 않습니다.

In [ ]:
!apt-get install -y fonts-nanum > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료. 런타임을 재시작하면 확실히 적용됩니다.')

## 셀 2 · 미로 알고리즘 정의

재귀적 백트래킹 생성기 + BFS 풀이 + 시각화 함수들.

In [ ]:
import random
from collections import deque
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation
from matplotlib.colors import ListedColormap

# 벽을 비트마스크로 표현 (한 칸 = 4비트: 위/오른쪽/아래/왼쪽)
TOP, RIGHT, BOTTOM, LEFT = 1, 2, 4, 8
DIRECTIONS = [
    (0, -1, TOP, BOTTOM),
    (1, 0, RIGHT, LEFT),
    (0, 1, BOTTOM, TOP),
    (-1, 0, LEFT, RIGHT),
]


@dataclass
class Maze:
    width: int
    height: int
    walls: np.ndarray = field(init=False)
    history: list = field(default_factory=list)

    def __post_init__(self):
        self.walls = np.full((self.height, self.width), 15, dtype=np.uint8)

    def carve(self, x1, y1, x2, y2):
        dx, dy = x2 - x1, y2 - y1
        for ddx, ddy, w_self, w_other in DIRECTIONS:
            if (ddx, ddy) == (dx, dy):
                self.walls[y1, x1] &= np.uint8(~w_self & 0xFF)
                self.walls[y2, x2] &= np.uint8(~w_other & 0xFF)
                return

    def has_wall(self, x, y, direction):
        return bool(self.walls[y, x] & direction)


def generate_maze(width, height, seed=None, record=False):
    """재귀적 백트래킹으로 미로 생성."""
    rng = random.Random(seed)
    maze = Maze(width, height)
    visited = np.zeros((height, width), dtype=bool)

    visited[0, 0] = True
    stack = [(0, 0)]

    while stack:
        x, y = stack[-1]
        candidates = []
        for dx, dy, _, _ in DIRECTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < width and 0 <= ny < height and not visited[ny, nx]:
                candidates.append((nx, ny))
        if candidates:
            nx, ny = rng.choice(candidates)
            maze.carve(x, y, nx, ny)
            visited[ny, nx] = True
            stack.append((nx, ny))
        else:
            stack.pop()
        if record:
            maze.history.append((visited.copy(), list(stack)))
    return maze


def solve_bfs(maze, start=None, end=None):
    """BFS로 최단 경로 탐색."""
    w, h = maze.width, maze.height
    if start is None: start = (0, 0)
    if end is None: end = (w - 1, h - 1)

    distances = np.full((h, w), -1, dtype=np.int32)
    distances[start[1], start[0]] = 0
    came_from = {start: None}
    queue = deque([start])

    while queue:
        x, y = queue.popleft()
        if (x, y) == end:
            break
        for dx, dy, w_dir, _ in DIRECTIONS:
            if maze.has_wall(x, y, w_dir):
                continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < w and 0 <= ny < h and distances[ny, nx] == -1:
                distances[ny, nx] = distances[y, x] + 1
                came_from[(nx, ny)] = (x, y)
                queue.append((nx, ny))

    if end not in came_from:
        return [], distances
    path = []
    cur = end
    while cur is not None:
        path.append(cur)
        cur = came_from[cur]
    path.reverse()
    return path, distances


def draw_maze(maze, path=None, ax=None, title=None):
    """matplotlib으로 미로를 그립니다."""
    w, h = maze.width, maze.height
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    ax.set_xlim(-0.5, w + 0.5)
    ax.set_ylim(h + 0.5, -0.5)
    ax.set_aspect('equal')
    ax.axis('off')

    for y in range(h):
        for x in range(w):
            cell = maze.walls[y, x]
            if cell & TOP:
                ax.plot([x, x + 1], [y, y], color='black', linewidth=1.5)
            if cell & LEFT:
                ax.plot([x, x], [y, y + 1], color='black', linewidth=1.5)
    ax.plot([w, w], [0, h], color='black', linewidth=1.5)
    ax.plot([0, w], [h, h], color='black', linewidth=1.5)

    ax.add_patch(plt.Rectangle((0, 0), 1, 1, color='#1D9E75', alpha=0.6))
    ax.add_patch(plt.Rectangle((w - 1, h - 1), 1, 1, color='#1D9E75', alpha=0.6))

    if path:
        xs = [p[0] + 0.5 for p in path]
        ys = [p[1] + 0.5 for p in path]
        ax.plot(xs, ys, color='#D85A30', linewidth=2.5, alpha=0.85)

    if title:
        ax.set_title(title, fontsize=13)
    return ax


def animate_generation(width, height, seed=None, interval=30):
    """미로 생성 과정 애니메이션."""
    maze = generate_maze(width, height, seed=seed, record=True)
    path, _ = solve_bfs(maze)

    fig, ax = plt.subplots(figsize=(8, 8))
    state = np.zeros((height, width), dtype=np.uint8)
    cmap = ListedColormap(['#FFFFFF', '#EEEDFE', '#CECBF6', '#7F77DD'])
    img = ax.imshow(state, cmap=cmap, vmin=0, vmax=3, extent=(0, width, height, 0))

    ax.set_xlim(-0.5, width + 0.5)
    ax.set_ylim(height + 0.5, -0.5)
    ax.set_aspect('equal')
    ax.axis('off')

    ax.add_patch(plt.Rectangle((0, 0), 1, 1, color='#1D9E75', alpha=0.4, zorder=1))
    ax.add_patch(plt.Rectangle((width-1, height-1), 1, 1, color='#1D9E75', alpha=0.4, zorder=1))

    title = ax.set_title('생성 중...', fontsize=13)

    # 최종 벽을 한 번만 그림
    for y in range(height):
        for x in range(width):
            cell = maze.walls[y, x]
            if cell & TOP:
                ax.plot([x, x+1], [y, y], color='black', linewidth=1.2, zorder=2)
            if cell & LEFT:
                ax.plot([x, x], [y, y+1], color='black', linewidth=1.2, zorder=2)
    ax.plot([width, width], [0, height], color='black', linewidth=1.2, zorder=2)
    ax.plot([0, width], [height, height], color='black', linewidth=1.2, zorder=2)

    path_line, = ax.plot([], [], color='#D85A30', linewidth=2.5, alpha=0, zorder=3)

    history = maze.history
    sample_step = max(1, len(history) // 150)
    sampled = list(range(0, len(history), sample_step))
    if sampled[-1] != len(history) - 1:
        sampled.append(len(history) - 1)
    frames = [('gen', i) for i in sampled] + [('path', i) for i in range(15)]

    def update(frame):
        kind, idx = frame
        if kind == 'gen':
            visited, stack_list = history[idx]
            state.fill(0)
            state[visited] = 1
            for sx, sy in stack_list:
                state[sy, sx] = 2
            if stack_list:
                tx, ty = stack_list[-1]
                state[ty, tx] = 3
            img.set_data(state)
            title.set_text(f'생성 중 · 스텝 {idx+1}/{len(history)}')
        else:
            alpha = min(1.0, (idx + 1) / 8)
            xs = [p[0] + 0.5 for p in path]
            ys = [p[1] + 0.5 for p in path]
            path_line.set_data(xs, ys)
            path_line.set_alpha(alpha)
            title.set_text(f'완성 · 최단 경로 {len(path)}칸')
        return img, path_line, title

    anim = animation.FuncAnimation(
        fig, update, frames=frames, interval=interval, blit=False, repeat=False
    )
    return fig, anim

print('알고리즘 로드 완료!')

## 셀 3 · 미로 만들고 풀기

`size`와 `seed`를 바꿔가며 다양한 미로를 만들어보세요. 같은 seed는 같은 미로를 만들고, seed=None이면 매번 다른 미로가 나옵니다.

In [ ]:
%matplotlib inline

size = 20
seed = 42

maze = generate_maze(size, size, seed=seed)
path, distances = solve_bfs(maze)

print(f'미로 크기: {size}×{size} (총 {size*size}칸)')
print(f'최단 경로 길이: {len(path)}칸')

fig, ax = plt.subplots(figsize=(8, 8))
draw_maze(maze, path=path, ax=ax, title=f'{size}×{size} 미로 (경로 {len(path)}칸)')
plt.show()

## 셀 4 · 여러 크기 한꺼번에 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, sz, sd in zip(axes, [10, 20, 30], [1, 7, 42]):
    m = generate_maze(sz, sz, seed=sd)
    p, _ = solve_bfs(m)
    draw_maze(m, path=p, ax=ax, title=f'{sz}×{sz} (경로 {len(p)}칸)')
plt.tight_layout()
plt.show()

## 셀 5 · 생성 과정 애니메이션

보라색이 현재 위치, 연보라가 스택(되돌아갈 경로), 흰색이 미방문 칸입니다.
마지막에 주황색으로 최단 경로가 그려져요.

In [ ]:
from IPython.display import HTML

fig, anim = animate_generation(15, 15, seed=1, interval=40)
plt.close(fig)  # 정적 이미지 중복 표시 방지
HTML(anim.to_jshtml())

## 셀 6 · 거리 히트맵 (보너스)

시작점에서 각 칸까지의 BFS 거리를 색으로 표현합니다.
어두울수록 가깝고, 밝을수록 멀리 있는 칸이에요.

In [ ]:
# 거리 맵 전체를 보려면 끝점 지정 없이 전체 BFS
size = 25
maze = generate_maze(size, size, seed=99)

# 끝점을 멀리 두면 BFS가 모든 칸을 탐색
_, distances = solve_bfs(maze, end=(-1, -1))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
draw_maze(maze, ax=axes[0], title='미로')
im = axes[1].imshow(distances, cmap='viridis')
axes[1].set_title('시작점으로부터의 거리')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], shrink=0.8, label='거리(칸)')
plt.tight_layout()
plt.show()

print(f'최장 거리: {distances.max()}칸')

## 다음 단계 아이디어

- `seed`를 `None`으로 바꿔 매번 다른 미로 생성
- `size`를 50, 100으로 키워보기 (애니메이션은 25 이하 권장)
- BFS 대신 A* 알고리즘 추가 구현
- Q-learning 에이전트로 미로 풀기 학습